### Carregando dependências

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score, confusion_matrix

In [3]:
df = pd.read_csv('./data/cogumelos_tratados.csv')
df.head()

,classe,forma_chapeu,superficie_chapeu,cor_chapeu,machucados,odor,fixacao_branquia,espacamento_branquia,tamanho_branquia,cor_branquia,...,superficie_caule_acima_anel,superficie_caule_abaixo_anel,cor_caule_acima_anel,cor_caule_abaixo_anel,cor_veu,numero_anel,tipo_anel,cor_impresao_esporos,populacao,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,s,w,w,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,s,w,w,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,s,w,w,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,s,w,w,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,s,w,w,w,o,e,n,a,g


### Preparando os dados

In [4]:
# separando as colunas
X = df.drop(columns='classe')

y = df['classe']

In [42]:
# código para pré-processar as colunas categoricas e transformar em númericas
features = X.columns

preprocessor = ColumnTransformer(transformers=[('categorical',
                                               OneHotEncoder(handle_unknown='ignore'),
                                               features)])

In [43]:
# dividindo os dados entre treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [44]:
# treinando o preprocessor
# pré-processando as colunas preditoras
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [8]:
print(X_train.shape)
print(X_test.shape)

(6499, 116)
(1625, 116)


### Construindo e treinando o modelo

In [ ]:
# instanciando o modelo de bagging
bagging_model = BaggingClassifier(
    estimator = LogisticRegression(),
    n_estimators = 5, # número de estimators (mini modelos)
    max_samples = 0.2, # porcentagem de registros que cada estimator pode usar
    max_features = 0.3, # porcentagem de features que o cada estimator pode usar
    bootstrap= False, # diz se pode ter ou não reposição de registro
    random_state = 42)

In [27]:
# treinando o modelo
bagging_model.fit(X_train, y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeClassifier`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",LogisticRegression()
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",5
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",0.2
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",0.3
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",False
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary <warm_start>`... versionadded:: 0.17 *warm_start* constructor parameter.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


### Fazendo a predição e avaliando

In [28]:
# fazendo a predição no conjunto de teste
y_pred = bagging_model.predict(X_test)

In [29]:
# mapeando os valores para númericos para fazer a avaliação
y_pred = pd.Series(y_pred)

mapping = {'e': 1, 'p': 0}

y_test_bin = y_test.map(mapping)
y_pred_bin = y_pred.map(mapping)

In [30]:
# métricas de avaliação do modelo
accuracy = accuracy_score(y_test_bin, y_pred_bin)
precision = precision_score(y_test_bin, y_pred_bin)
recall =  recall_score(y_test_bin, y_pred_bin)
f1 =  f1_score(y_test_bin, y_pred_bin)

print(f'    Acurácia: {accuracy:.3f}')
print(f'    Precisão:{precision:.3f}')
print(f'    Recall:{recall:.3f}')
print(f'    F1:{f1:.3f}')

    Acurácia: 0.993
    Precisão:0.995
    Recall:0.992
    F1:0.993


In [31]:
# registros que cada estimator usou
bagging_model.estimators_samples_

[array([6332, 4473, 2897, ...,  496, 6017, 2089], shape=(1299,)),
 array([  62, 4060, 5066, ..., 5535, 1335, 6433], shape=(1299,)),
 array([ 191, 3451, 5754, ..., 4370, 5716, 2113], shape=(1299,)),
 array([ 864,  974, 6383, ...,  967, 5381, 6448], shape=(1299,)),
 array([6296, 6407, 1997, ...,  104, 3535,  755], shape=(1299,))]

In [32]:
#features que cada estimator usou
bagging_model.estimators_features_

[array([ 48, 111,  54, 115,  27,  85,  24,  39,  31,  26,  10,  30,  99,
         79,  50, 101,  65,  40,  69,  16,  67,  15,  83,  72,  35,  73,
         29, 105,  43,   5,  95,  12,  98, 104]),
 array([ 62,  48,  77,  67,  64,  52,  26,  38,  37,  70,  13,  36,  66,
         29,   8,  21, 111,  96,  27,  34, 104,   4,  32,  28, 110,  86,
         22,  93,  92,  20,  78,   5,  97, 102]),
 array([ 30,  98,  50, 112,   0, 113,   1,  23, 108,  43,   6,  47,  21,
         42,  72, 102, 104,  24,  75,  39, 100,  93,  63,   4,  67, 110,
         31,  78,   9,  99,  35,  90,  57,  26]),
 array([ 29,  68,  47,  46,  75,  99,  64,  94, 110,  21,  19,  39,  45,
        104,  30,  96,  52, 109,  16,  14, 103,  97,  40,  73,  81, 101,
         11,  15, 100, 114,  89,  13, 113,  90]),
 array([ 31,  65,  72,  63,   7,  22,  45,  43,  78,  91,  89,  17,  16,
          6,  66,  26,  28,  36, 115,  21,  44,  41, 106,  13,  33,  95,
         11,  20,  42,  27,  32,  93,  12,  52])]

In [33]:
# matriz para visualização dos valores de acerto e erro

matrix = confusion_matrix(y_test_bin, y_pred_bin)

fig = px.imshow(matrix,
                labels=dict(x='Predição', y='Real', color='Contagem'),
                x=['Venenoso', 'Comestível'],
                y=['Venenoso', 'Comestível'])

fig.update_traces(text=matrix, texttemplate='%{z}')
fig.update_layout(coloraxis_showscale=False, height=350, width=400)
fig.show()

In [ ]:
# esse método vai retornar a probabilidade do valor verdadeiro ser "e" ou "p" (nessa ordem)
y_pred_prob = bagging_model.predict_proba(X_test)
y_pred_prob

array([[0.96000785, 0.03999215],
       [0.01612107, 0.98387893],
       [0.06779717, 0.93220283],
       ...,
       [0.03318506, 0.96681494],
       [0.01242779, 0.98757221],
       [0.0187335 , 0.9812665 ]], shape=(1625, 2))

### Salvando o modelo

In [34]:
# salvando o modelo
import joblib

joblib.dump(bagging_model, './model/bagging_model.pkl')

['./model/bagging_model.pkl']

### Código elaborado com max_features = 1

In [27]:
# calculando o coeficiente médio de cada coluna
importance = np.mean([np.abs(estimador.coef_[0]) for estimador in bagging_model.estimators_], axis=0)

# buscando o nome de cada coluna depois do OneHotEncoder
features_name = (preprocessor.named_transformers_['categorical'].get_feature_names_out(features).tolist())

In [28]:
# criando um dataframe com o nome e o coeficiente médio de cada coluna
df_FeaturesImportance = pd.DataFrame({'Feature': features_name, 'Importance': importance})
df_FeaturesImportance = df_FeaturesImportance.sort_values(ascending=True, by='Importance')

In [29]:
# gráfico para visualizar a coluna com mais importância no modelo
fig = px.bar(data_frame=df_FeaturesImportance,
                x='Importance',
                y='Feature',
                orientation='h',
                title='A importância de cada coluna para o modelo (baseado no valor médio absoluto)')

fig.update_layout(height=1500, width=1000)
fig.show()